# Session 6 – Build an agent: tools, a loop and a supervisor
An agent is a model in a loop: **goal → choose a tool → act → read the result → repeat → stop**.
The tools here are real (ClinicalTrials.gov, PubMed); sending e-mail is simulated and only writes to a file.
The patient is a published case report (Scandurra et al., *EMJ Oncology* 2023, CC BY-NC 4.0), so everything sent to the model is public (Tier 1).

In [ ]:
import os, sys, json, time, getpass, subprocess, urllib.request, urllib.parse
try:
    from openai import OpenAI
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "openai"])
    from openai import OpenAI

## 1. Connect
The same NVIDIA key as in Sessions 4 and 5. The fastest model that follows the format is tried first.

In [ ]:
API_KEY = os.environ.get("NVIDIA_API_KEY") or getpass.getpass("NVIDIA API key: ")
client = OpenAI(base_url="https://integrate.api.nvidia.com/v1", api_key=API_KEY, timeout=60, max_retries=1)

MODEL = None                                   # or set by hand
CANDIDATES = ["nvidia/nemotron-3-super-120b-a12b", "google/gemma-4-31b-it", "openai/gpt-oss-20b"]

def works(m):
    try:
        t = time.time()
        client.chat.completions.create(model=m, messages=[{"role": "user", "content": "Hi"}], max_tokens=5)
        print(f"  {m}: OK ({time.time() - t:.1f} s)"); return True
    except Exception as e:
        print(f"  {m}: not available ({str(e)[:60]})"); return False

MODEL = MODEL or next((m for m in CANDIDATES if works(m)), None)
print("Using:", MODEL)

## 2. The patient and the goal

In [ ]:
PATIENT = """67-year-old woman living in Catania, Italy.
Endometrioid endometrial adenocarcinoma, grade 3, FIGO stage IIIC.
Immunohistochemistry: MLH1 and PMS2 lost, MSH2 and MSH6 retained. MSI-high by NGS (the PCR test was negative). TMB 12.9 mutations/Mb.
Treatment so far: hysterectomy, removal of both adnexa and pelvic lymph nodes; 2 cycles of carboplatin + paclitaxel, stopped when a scalp metastasis appeared (removed);
a brain metastasis (left temporal-parietal) treated with radiotherapy, 42 Gy; then dostarlimab (anti-PD-1).
Other metastases: left adrenal, pelvic lymph nodes. Performance status 2-3 when dostarlimab started, 1 at the last report.
Not reported: germline (Lynch syndrome) testing, MLH1 promoter methylation."""
# Source: Scandurra G. et al. EMJ Oncol. 2023;11(1):98-105, doi:10.33590/emjoncol/10300406

GOAL = ("Find recruiting clinical trials in Italy that this patient might join. Check her age against each candidate trial's limits. "
        "Then draft an e-mail to the contact person of the best match.")

## 3. Build the tools (with your AI assistant)
Each tool is an ordinary Python function. The model never runs code: it only asks for a tool by name, and your function does the work.
For each tool below: give your AI assistant the specification, review what it writes (Session 2!), paste it in, and run the test until it prints `TEST: True`.
Rules: use only Python's standard library (`urllib`, `json`), keep the exact function name and inputs, and return **text**.

In [ ]:
import json, urllib.request, urllib.parse

def _get(url):
    """Download a URL and read it as JSON (used by the tools below)."""
    with urllib.request.urlopen(url, timeout=20) as r:
        return json.loads(r.read())

### Tool 1: `search_trials(condition, location="")`
- **Does:** searches ClinicalTrials.gov for **recruiting** trials of a condition, optionally in a country or city.
- **Data source:** the ClinicalTrials.gov API, version 2 (`https://clinicaltrials.gov/api/v2/studies`); documentation at https://clinicaltrials.gov/data-api/api
- **Returns:** text, one line per trial (at most 5): `NCT number: title (phase)`, or `"No recruiting trials found."`

In [ ]:
def search_trials(condition, location=""):
    """Recruiting trials on ClinicalTrials.gov: one line per trial, 'NCT...: title (phase)'."""
    q = {"query.cond": condition, "filter.overallStatus": "RECRUITING", "pageSize": 5}
    if location:
        q["query.locn"] = location
    studies = _get("https://clinicaltrials.gov/api/v2/studies?" + urllib.parse.urlencode(q)).get("studies", [])
    if not studies:
        return "No recruiting trials found."
    lines = []
    for s in studies:
        p = s["protocolSection"]
        phases = ", ".join(p.get("designModule", {}).get("phases", [])) or "n/a"
        lines.append(f'{p["identificationModule"]["nctId"]}: {p["identificationModule"]["briefTitle"]} ({phases})')
    return "\n".join(lines)

In [ ]:
r = search_trials("endometrial cancer", "Italy")
print(r)
print("TEST:", isinstance(r, str) and ("NCT" in r or "No recruiting" in r))

### Tool 2: `get_trial(nct_id)`
- **Does:** fetches one trial's details from ClinicalTrials.gov (`https://clinicaltrials.gov/api/v2/studies/NCT...`).
- **Returns:** text with these labelled lines, so the model can find them:
  `Minimum age: ...` · `Maximum age: ...` · `Sex: ...` · `Contact: name <e-mail>` · `Eligibility:` followed by the first ~1,200 characters of the eligibility criteria.

In [ ]:
def get_trial(nct_id):
    """One trial's age limits, sex, contact and eligibility criteria, as labelled lines of text."""
    p = _get(f"https://clinicaltrials.gov/api/v2/studies/{nct_id}")["protocolSection"]
    e = p.get("eligibilityModule", {})
    contacts = p.get("contactsLocationsModule", {}).get("centralContacts", [])
    contact = f'{contacts[0].get("name", "?")} <{contacts[0].get("email", "no e-mail listed")}>' if contacts else "none listed"
    return (f'{nct_id}: {p["identificationModule"]["briefTitle"]}\n'
            f'Minimum age: {e.get("minimumAge", "N/A")}\nMaximum age: {e.get("maximumAge", "N/A")}\nSex: {e.get("sex", "ALL")}\n'
            f'Contact: {contact}\nEligibility:\n{e.get("eligibilityCriteria", "")[:1200]}')

In [ ]:
import re
first = re.search(r"NCT\d{8}", search_trials("endometrial cancer", "Italy"))
r = get_trial(first.group() if first else "NCT03981796")
print(r[:600])
print("TEST:", all(label in r for label in ["Minimum age:", "Maximum age:", "Contact:", "Eligibility"]))

### Tool 3: `check_age(patient_age, minimum_age, maximum_age)`
- **Does:** compares the patient's age with a trial's limits. The limits arrive as ClinicalTrials.gov writes them: `"18 Years"`, `"6 Months"`, or `"N/A"` (no limit).
- **Returns:** exactly one of `"eligible"`, `"too young"`, `"too old"`. No internet needed.

In [ ]:
def check_age(patient_age, minimum_age, maximum_age):
    """'eligible', 'too young' or 'too old'. Limits are text like '18 Years' or 'N/A'."""
    def years(text):
        if not text or str(text).upper() == "N/A":
            return None
        number = float(str(text).split()[0])
        return number / 12 if "month" in str(text).lower() else number
    low, high = years(minimum_age), years(maximum_age)
    if low is not None and float(patient_age) < low:
        return "too young"
    if high is not None and float(patient_age) > high:
        return "too old"
    return "eligible"

In [ ]:
tests = [((67, "18 Years", "N/A"), "eligible"), ((67, "18 Years", "65 Years"), "too old"),
         ((16, "18 Years", "N/A"), "too young"), ((1, "6 Months", "N/A"), "eligible")]
for args, expected in tests:
    print(args, "->", check_age(*args), "| expected", expected)
print("TEST:", all(check_age(*a) == e for a, e in tests))

### Tool 4: `search_pubmed(query)`
- **Does:** finds the 3 most relevant PubMed papers for a query.
- **Data source:** NCBI E-utilities: `esearch` to get PubMed IDs, then `esummary` to get their titles; documentation at https://www.ncbi.nlm.nih.gov/books/NBK25499/
- **Returns:** text, one line per paper: `PMID 12345678: title (journal, year)`, or `"No papers found."`

In [ ]:
def search_pubmed(query):
    """3 relevant PubMed papers: one line each, 'PMID ...: title (journal, year)'."""
    base = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"
    ids = _get(base + "esearch.fcgi?" + urllib.parse.urlencode(
        {"db": "pubmed", "term": query, "retmode": "json", "retmax": 3, "sort": "relevance"}))["esearchresult"]["idlist"]
    if not ids:
        return "No papers found."
    summ = _get(base + "esummary.fcgi?" + urllib.parse.urlencode({"db": "pubmed", "id": ",".join(ids), "retmode": "json"}))["result"]
    return "\n".join(f'PMID {i}: {summ[i]["title"]} ({summ[i].get("source", "")}, {summ[i].get("pubdate", "")[:4]})' for i in ids)

In [ ]:
r = search_pubmed("dostarlimab mismatch repair deficient endometrial cancer")
print(r)
print("TEST:", isinstance(r, str) and ("PMID" in r or "No papers" in r))

### Tool 5: `send_email(to, subject, text)` – an **action**
- **Does:** *simulates* sending an e-mail. It must **never** really send anything: it appends the e-mail to a file called `outbox.txt`.
- **Returns:** a short confirmation, e.g. `"E-mail to ... written to outbox.txt."`

In [ ]:
def send_email(to, subject, text):
    """Simulated: append the e-mail to outbox.txt (never really send). Return a confirmation."""
    with open("outbox.txt", "a", encoding="utf-8") as f:
        f.write(f"TO: {to}\nSUBJECT: {subject}\n{text}\n{'=' * 60}\n")
    return f"E-mail to {to} written to outbox.txt."

In [ ]:
r = send_email("test@example.org", "TEST-SUBJECT", "Hello")
print(r)
print("TEST:", "TEST-SUBJECT" in open("outbox.txt", encoding="utf-8").read())
open("outbox.txt", "w").close()                 # empty the outbox again before the agent runs

### Stuck on a tool?
After about 25 minutes, load working versions of all five tools from the file next to this notebook and move on to the agent. (This replaces your versions.)

In [ ]:
# from agent_tools_ready import *

## 4. Register the tools and decide the permissions
`read` tools only look things up. `act` tools change something in the world: with the supervisor on, they need your approval. Change a `kind` and see what happens.

In [ ]:
TOOLS = {
    "search_trials": {"fn": search_trials, "kind": "read", "about": "search_trials(condition, location): recruiting trials on ClinicalTrials.gov"},
    "get_trial":     {"fn": get_trial,     "kind": "read", "about": "get_trial(nct_id): age limits, sex, eligibility criteria and contact of one trial"},
    "search_pubmed": {"fn": search_pubmed, "kind": "read", "about": "search_pubmed(query): titles of 3 relevant papers"},
    "check_age":     {"fn": check_age,     "kind": "read", "about": "check_age(patient_age, minimum_age, maximum_age): eligible / too young / too old"},
    "send_email":    {"fn": send_email,    "kind": "act",  "about": "send_email(to, subject, text): send an e-mail"},
}
MAX_STEPS = 10                                  # the agent must stop after this many steps

## 5. The loop
The whole agent: ask the model for the next step, run the tool, give back the result, repeat.

In [ ]:
def system_prompt():
    tools = "\n".join(f"- {t['about']}" for t in TOOLS.values())
    return f"""You are an agent that helps find clinical trials for a patient. You can use these tools:
{tools}
Reply with ONLY one line of JSON for your next step, no explanations, for example:
{{"tool": "search_trials", "input": {{"condition": "endometrial cancer", "location": "Italy"}}}}
You have at most {MAX_STEPS} steps. When you are done, reply {{"tool": "finish", "input": {{"answer": "your final answer"}}}}"""

THINKING_OFF = True              # reasoning models: ask for the answer without long hidden "thinking"

def parse_call(text):
    """The last line of JSON in the text that names a tool, as (tool, input); None if there is none."""
    for line in reversed((text or "").splitlines()):
        if '"tool"' in line and "{" in line:
            try:
                call = json.loads(line[line.find("{"): line.rfind("}") + 1])
                return call["tool"], call.get("input", {})
            except Exception:
                pass
    try:
        call = json.loads(text[text.find("{"): text.rfind("}") + 1])
        return call["tool"], call.get("input", {})
    except Exception:
        return None

def next_step_from_model(messages):
    global THINKING_OFF
    t = time.time()
    kw = dict(model=MODEL, messages=messages, temperature=0, max_tokens=1500)
    try:
        r = client.chat.completions.create(**kw, extra_body={"chat_template_kwargs": {"enable_thinking": False}}) if THINKING_OFF             else client.chat.completions.create(**kw)
    except Exception as e:
        if not THINKING_OFF or "timed out" in str(e).lower():
            raise
        THINKING_OFF = False                    # this model does not accept the setting: ask normally
        r = client.chat.completions.create(**kw)
    msg = r.choices[0].message
    text = (msg.content or "").strip()
    if parse_call(text) is None and getattr(msg, "reasoning_content", None):
        text = msg.reasoning_content.strip()    # the answer sometimes stays inside the model's "thinking"
    return text, time.time() - t

def ask_human(step):
    while True:
        reply = input(f"Step {step} - your next step as JSON (q = stop): ").strip()
        if reply:
            return reply
        print("  Type one line of JSON, e.g. " + '{"tool": "get_trial", "input": {"nct_id": "NCT06330064"}}')

def run_agent(goal, patient, supervisor=False, you_are_the_model=False, label="run", quiet=False):
    messages = [{"role": "system", "content": system_prompt()},
                {"role": "user", "content": f"{goal}\n\nPATIENT:\n{patient}"}]
    trace = []
    say = (lambda *a: None) if quiet else print
    if you_are_the_model:
        say("You are the model. Tools:\n" + "\n".join(f"  {t['about']}" for t in TOOLS.values())
            + '\nFinish with {"tool": "finish", "input": {"answer": "..."}}\n')
    for step in range(1, MAX_STEPS + 1):
        if you_are_the_model:
            reply, secs = ask_human(step), 0
            if reply.lower() in ("q", "quit", "exit", "stop"):
                say("Stopped by you."); break
        else:
            try:
                reply, secs = next_step_from_model(messages)
            except Exception as e:
                say(f"[{step}] the model did not answer: {str(e)[:100]}")
                trace.append({"step": step, "result": f"model error: {str(e)[:100]}"}); continue
        call = parse_call(reply)
        if call is None:
            say(f"[{step}] reply was not one line of JSON: {reply[:100] or '(empty)'}")
            if reply:
                messages.append({"role": "assistant", "content": reply})
            messages.append({"role": "user", "content": 'Your reply was empty or not JSON. Reply with ONE line of JSON only, e.g. {"tool": "finish", "input": {"answer": "..."}}'})
            trace.append({"step": step, "reply": reply, "result": "format error"}); continue
        tool, args = call
        say(f"\n[{step}] {tool} {json.dumps(args)[:150]}  ({secs:.1f} s)")
        if tool == "finish":
            say(f"FINISHED: {args.get('answer', '')}")
            trace.append({"step": step, "tool": tool, "input": args}); break
        if tool not in TOOLS:
            result = f"There is no tool called {tool}."
        elif supervisor and TOOLS[tool]["kind"] == "act" and input(f"SUPERVISOR: allow {tool}({json.dumps(args)[:300]})? y/n ").strip().lower() != "y":
            result = "The human supervisor did NOT approve this action."
        else:
            try:
                result = str(TOOLS[tool]["fn"](**args))
            except Exception as e:
                result = f"Tool error: {e}"
        say("   -> " + result[:300].replace("\n", "\n      "))
        messages += [{"role": "assistant", "content": json.dumps({"tool": tool, "input": args})},
                     {"role": "user", "content": f"RESULT:\n{result[:1500]}"}]
        trace.append({"step": step, "tool": tool, "input": args, "result": result[:1500], "seconds": round(secs, 1)})
    else:
        say(f"\nStopped: reached MAX_STEPS = {MAX_STEPS}")
    os.makedirs("traces", exist_ok=True)
    json.dump(trace, open(f"traces/{label}.json", "w"), indent=1, ensure_ascii=False)
    return trace

## 6. Round 1 – no supervisor
Let it run on its own. Then read the outbox: what was "sent", to whom, and was it right?

In [ ]:
trace1 = run_agent(GOAL, PATIENT, supervisor=False, label="round1")
print("\nOUTBOX:\n" + (open("outbox.txt", encoding="utf-8").read() if os.path.exists("outbox.txt") else "(empty)"))

## 7. Round 2 – supervisor on
The same goal. Before every `act` step you are asked to approve. What would make you say no?

In [ ]:
trace2 = run_agent(GOAL, PATIENT, supervisor=True, label="round2")

## 8. You are the model
Now you choose each step, as in the role-play. Type one JSON line per step, e.g.
`{"tool": "search_trials", "input": {"condition": "endometrial cancer", "location": "Italy"}}`

In [ ]:
trace3 = run_agent(GOAL, PATIENT, supervisor=True, you_are_the_model=True, label="you")

## 9. Bias check – change one thing about the patient
Same goal without the e-mail, run twice: once as published, once with one detail changed. Do the trials it chooses change? Should they?

In [ ]:
SWAP = ("67-year-old", "45-year-old")          # try also: ("Performance status 2-3", "Performance status 0")
GOAL_LIST = "Find recruiting clinical trials in Italy that this patient might join, check her age for each, and finish with a list of NCT numbers. Do not send e-mails."
runs = {}
for name, patient in [("as published", PATIENT), (f"{SWAP[0]} -> {SWAP[1]}", PATIENT.replace(*SWAP))]:
    tr = run_agent(GOAL_LIST, patient, label=name.replace(" ", "_").replace(">", ""), quiet=True)
    runs[name] = tr[-1].get("input", {}).get("answer", "(did not finish)") if tr else "(no steps)"
for name, answer in runs.items():
    print(f"=== {name}\n{answer}\n")

## 10. Replay a saved run
Every run is saved in the `traces` folder. Replay one step by step.

In [ ]:
def show_trace(label):
    for t in json.load(open(f"traces/{label}.json", encoding="utf-8")):
        print(f"[{t['step']}] {t.get('tool', '?')} {json.dumps(t.get('input', {}))[:150]}")
        if t.get("result"):
            print("   -> " + t["result"][:200].replace("\n", " "))

show_trace("round1")